# Learning-to-rank fine-tuning of E5 on Colab

Runtime -> Change runtime type -> **T4 GPU**. Total time is about 2-2.5 hours. Run the cells in order; every step writes to Google Drive, so a disconnect only loses the step in progress.

Required on Drive at `MyDrive/DATA304_final_project`:
- the latest `scripts/` and `src/`
- `project_release/` (`catalog.json`, `train_gold.json`, `test_queries.json`)
- `outputs/ltr/negatives_dev.json` and `outputs/ltr/negatives_full.json` (created locally by `scripts/mine_negatives.py`; they contain item IDs only)

No review text is sent to any LLM service. Training and encoding run inside this Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/DATA304_final_project'
%cd $PROJECT
!ls scripts src project_release outputs/ltr

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.40"
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## Step 1: model A (validation model)
It trains on 12,000 development-training queries. Each epoch is scored on the 4,000-query tuning slice, and the best epoch is saved. About 60 minutes.

In [ ]:
!python scripts/train_dense_ltr.py --negatives outputs/ltr/negatives_dev.json --output-dir outputs/ltr/e5-ltr-dev

In [ ]:
import json
log = json.load(open('outputs/ltr/e5-ltr-dev/train_log.json'))
for e in log['epochs']: print(e)
K = log['selected_epoch']; print('selected epoch:', K)

## Step 2: encode with model A (catalog and training reviews, used for local validation)
About 9 minutes.

In [ ]:
!python scripts/encode_dense.py --model outputs/ltr/e5-ltr-dev/checkpoint --tag e5-ltr-dev --parts catalog,train

## Step 3: model B (final model, all 20,000 training queries, same schedule, stops at epoch K)
About 20 minutes per epoch.

In [ ]:
!python scripts/train_dense_ltr.py --negatives outputs/ltr/negatives_full.json --output-dir outputs/ltr/e5-ltr-full --stop-after-epoch {K}

## Step 4: encode with model B (catalog and test reviews, used for the Kaggle file)
About 9 minutes.

In [ ]:
!python scripts/encode_dense.py --model outputs/ltr/e5-ltr-full/checkpoint --tag e5-ltr-full --parts catalog,test
!ls -la outputs/dense/e5-ltr-dev outputs/dense/e5-ltr-full

Download these folders into the same paths in the local project:
- `outputs/dense/e5-ltr-dev/` (catalog.npy, train_reviews.npy, manifest.json)
- `outputs/dense/e5-ltr-full/` (catalog.npy, test_reviews.npy, manifest.json)
- `outputs/ltr/e5-ltr-dev/train_log.json` and `outputs/ltr/e5-ltr-full/train_log.json`

Keep the `checkpoint/` folders on Drive (about 440 MB each). The final README will link to the model B checkpoint for reproduction.